# Radially symmetric off-axis PSFs
The most basic format for off-axis PSFs provided by a yield input package is a radially symmetric one. 

In [ ]:
# Open the native yield input package
from yippy import open_yip

package = open_yip("usort_offaxis_ovc")

In [ ]:
# Read the shipped PSFs and their native XY positions.
import matplotlib.pyplot as plt
import numpy as np
lod = "lambda/D"  # Plot label; coordinates and diameters are floats.

native_psfs = package.offaxis_psfs
offsets_xy_lod = package.offsets_xy_lod

## The offsets
Each native XY pair is shown at its delivered position in lambda/D. This package samples only the x-axis.

In [ ]:
n_offsets = offsets_xy_lod.shape[0]
scatter = plt.scatter(*offsets_xy_lod.T, c=np.arange(n_offsets))
plt.title("Off-axis PSF offsets")
plt.xlabel(f"x ({lod})")
plt.ylabel(f"y ({lod})")
plt.colorbar(scatter, label="Offset's index value")
plt.show()

## The PSFs
Now, lets look at what the psfs included in the yield input package look like. Given that all the offsets are along the x-axis we would assume that the PSFs show a point source moving along the x-axis as well.

In [ ]:
# Utility functions
def set_up_plot():
    rows, cols = 4, 4
    base_size = 3
    fig, axes = plt.subplots(
        nrows=rows, ncols=cols, figsize=(base_size * cols, base_size * rows)
    )
    return fig, axes


def ax_labels_and_ticks(ax):
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel("y (pixel)")
    else:
        ax.set_yticks([])
    if ax.get_subplotspec().is_last_row():
        ax.set_xlabel("x (pixel)")
    else:
        ax.set_xticks([])
    return ax

In [ ]:
### Plotting PSF data
fig, axes = set_up_plot()

# Choosing the PSF offset index values
offsets = np.linspace(
    0, len(offsets_xy_lod) - 1, axes.shape[0] * axes.shape[1], dtype=int
)

# Loop through and plot the PSFs
for ax, offset in zip(axes.flatten(), offsets):
    # Get the x, y values of the offset from the offset list
    offset_x, offset_y = offsets_xy_lod[offset]

    # Load the PSF from the provided data
    psf = native_psfs[offset]

    # Plot the PSF data at the current offset
    ax.imshow(psf, origin="lower")

    # Make the plot pretty
    ax.set_title(f"Offset {offset}: ({offset_x:.1f}, {offset_y:.1f}) {lod}")
    ax = ax_labels_and_ticks(ax)
plt.show()

## Using yippy
I would assume you don't want to be limited to these specific PSFs that are all on the positive x-axis. Luckily, the major use-case for yippy is to handle the interpolation for you!

For a 1D YIP, yippy interpolates between the tabulated PSFs at the requested radial separation and translates the result to the requested $(x, y)$ position. It does not rotate the PSF to the position angle: rotation would also rotate the diffraction structure set by the pupil (spiders, segment gaps), which stays fixed on the detector, and gives unrealistic PSFs. As a result, any azimuthal structure in the tabulated PSFs (for example, the outward shift of the PSF core near the inner working angle) keeps its +x orientation at every position angle. A 1D grid does not sample that azimuthal variation, so no interpolation can recover it; use a 2D YIP for spatially resolved image simulation.

In [ ]:
# yippy has a lot of logging messages, disabling to keep this documentation clean
import logging


yippy_logger = logging.getLogger("yippy")
yippy_logger.setLevel(logging.ERROR)

In [ ]:
coro = package.prepare()

In [ ]:
### Plotting PSF data
fig, axes = set_up_plot()

# Choosing the PSF offset index values
n_plots = axes.shape[0] * axes.shape[1]

x_offsets = np.linspace(0, np.linalg.norm(offsets_xy_lod, axis=1).max(), n_plots)
y_offsets = np.zeros(n_plots)

# Loop through and plot the PSFs
for ax, x, y in zip(axes.flatten(), x_offsets, y_offsets):
    # USE YIPPY TO GENERATE THE PSF
    ######################
    psf = coro.source_psfs(None, x, y, grid=None).image.integrated().data[0]
    ######################

    # Plot the PSF data at the current offset
    ax.imshow(psf, origin="lower")

    # Make the plot pretty
    ax.set_title(f"({x:.1f}, {y:.1f}) {lod}")
    ax = ax_labels_and_ticks(ax)
fig.suptitle(f"Radially symmetric {package.source.path.name}")
plt.show()

Ok, it isn't an exact match because the offsets in the YIP were not spaced linearly (nor do they have to be for sampling reasons) and I used `np.linspace`. However, `yippy` interpolated between the provided offsets PSFs which is what we want! I should note here that as with any interpolation, we are at the mercy of the data given to the interpolant. If the offsets are poorly sampled there is nothing that can be done to fill in the missing information.

## Interpolation in 2D
Lets make a quick plot of random 2D offsets in $\lambda/D$ units (with logarithmic normalization).

In [ ]:
from matplotlib.colors import LogNorm
### Plotting PSF data
fig, axes = set_up_plot()

# Choosing the PSF offset index values
n_plots = axes.shape[0] * axes.shape[1]

x_offsets = np.random.uniform(low=-15, high=15, size=(n_plots,))
y_offsets = np.random.uniform(low=-15, high=15, size=(n_plots,))

# Loop through and plot the PSFs
for ax, x, y in zip(axes.flatten(), x_offsets, y_offsets):
    # USE YIPPY TO GENERATE THE PSF
    ######################
    psf = coro.source_psfs(None, x, y, grid=None).image.integrated().data[0]
    ######################

    # Plot the PSF data at the current offset
    ax.imshow(psf, origin="lower", norm=LogNorm())

    # Make the plot pretty
    ax.set_title(f"({x:.1f}, {y:.1f}) {lod}")
    ax = ax_labels_and_ticks(ax)
fig.suptitle(f"Radially symmetric {package.source.path.name}")
plt.show()